# Akida ECG port — first slice (see docs/akida_ecg_results.md)

Thin notebook: all logic lives in `eia.akida_models` and `scripts/akida_verify.py`.
**Linux only** — BrainChip's `akida` package has no macOS wheel (ever, any release; see
`Dockerfile.akida`). This notebook works on **Google Colab** (Linux runtime) or inside the
repo's Docker container; it will not run on a local macOS Jupyter kernel.

Ports the ECG modality (the one modality already proven to genuinely learn, on real
MIT-BIH) onto BrainChip MetaTF: a small quantized Conv2D-over-time CNN, quantized via
`quantizeml`, converted via `cnn2snn`, run on the Akida **software simulator** (no
hardware needed). Measures whether the float→on-chip fidelity gap shrinks vs. the known
Xylo number (float 0.845 balanced acc → XyloSim ~0.56) — see `docs/akida_ecg_results.md`
for the full write-up this notebook reproduces a slice of.

## Setup
On Colab (Linux) this clones the repo and installs `eia[data,akida]` — MetaTF's exact
pinned versions (see `pyproject.toml`'s `akida` extra) to avoid the resolver backtracking
that made an unconstrained install impractically slow (`docs/akida_ecg_results.md` Part 0).
Locally, it only proceeds if `akida` is already importable (i.e. you're running this
kernel *inside* the Docker container: `scripts/akida_docker_run.sh jupyter notebook`).

In [ ]:
import importlib.util, sys, subprocess, os

IN_COLAB = 'google.colab' in sys.modules

if importlib.util.find_spec('eia') is None:
    if IN_COLAB:
        # EDIT to your GitHub repo URL after you push:
        REPO = 'https://github.com/USERNAME/eia.git'
        subprocess.run(['git', 'clone', REPO], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'eia[data,akida]'],
                       check=True)
        sys.path.insert(0, 'eia/src')
        sys.path.insert(0, 'eia/scripts')
    else:
        sys.path.insert(0, os.path.abspath('../src'))
        sys.path.insert(0, os.path.abspath('../scripts'))

if importlib.util.find_spec('akida') is None:
    raise RuntimeError(
        "'akida' is not importable. This notebook needs Linux (Colab) or the repo's "
        "Docker container -- see Dockerfile.akida / README.md's 'Verify on Akida' section. "
        "It cannot run on a local macOS kernel (no macOS wheel exists for 'akida', ever).")

import eia
print('eia', eia.__version__)

## Part 0 — confirm the toolchain (versions, software backend, no hardware)

In [ ]:
import akida, cnn2snn, quantizeml, tensorflow as tf

print('akida       ', akida.__version__)
print('quantizeml  ', quantizeml.__version__)
print('cnn2snn     ', cnn2snn.__version__)
print('tensorflow  ', tf.__version__)
print('akida target', cnn2snn.get_akida_version())
print('devices     ', akida.devices(), '(empty = no hardware attached, software sim only)')

## Load real MIT-BIH ECG
Same `eia.datasets.load_ecg` real-data loader the Xylo path uses — needs `wfdb` + network.

In [ ]:
from eia.datasets import load_ecg
from eia import report

data = load_ecg(prefer_real=True)
card = report.data_card(data)

## Train (float) → quantize + QAT fine-tune → convert → verify against the Akida sim
Reuses `scripts/akida_verify.train_and_verify` — the exact same function the multi-seed CLI
measurement (`python scripts/akida_verify.py --real --n-seeds 5`) calls, so this notebook
and `docs/akida_ecg_results.md`'s numbers come from one code path, not two.

In [ ]:
import akida_verify

result = akida_verify.train_and_verify(
    data, seed=0, epochs=15, qat_epochs=5, n_restarts=3, max_verify=300,
    weight_bits=8, activation_bits=8)
akida_verify.print_result(result)

## Next
- `python scripts/akida_verify.py --real --n-seeds 5` for the full multi-seed measurement
  (mean ± spread — a single seed is not reliable evidence on its own, the same lesson the
  ECG/PPG Xylo quantization work learned; see `docs/ecg_quant_fixes_results.md`).
- `docs/akida_ecg_results.md` for the full Part-0 findings, the chosen architecture's
  rationale, and the Xylo-gap comparison verdict.
- The Rockpool/XyloSim path (`scripts/xylo_verify.py`) is untouched — this is a parallel
  deploy sibling, not a replacement.